# 05 · Verifier (lever 4)

**What changed (v3 to v4):** `nestling/verifier.py` checks every claim in a medical, diet or insurance answer against its cited passages. One rewrite is allowed; if claims are still unsupported the answer becomes *"I don't have a trusted source for that. Please ask your doctor."*

**Hypothesis (plan §9.5 #4):** faithfulness +5 points, latency +1 to 2 s.

v5 is the round-2 version after failure analysis (code-only unsafe kinds, Concierge v2, source-diversity cap, single-rewrite Verifier). It is shown here because it changes what the Verifier costs.

In [1]:
from evals import compare

table = compare.comparison_table()
BEFORE, AFTER = "v3", "v4"
rows = [r for r in table if r["version"] in (BEFORE, AFTER)]
print(compare.format_table(rows))

version                 esc             refuse              route               cite              faith              p95 s            mean Rs             p95 Rs
v3           100.0% (+0.0%)    99.0% (+0.0%) X    82.7% (-1.9%) X      78.1% (-2.7%)    76.1% (-3.3%) X    10.55 (+2.24) X     0.951 (+0.034)     1.555 (+0.014)
v4           100.0% (+0.0%)    99.0% (+0.0%) X    81.7% (-1.0%) X     67.1% (-11.0%)     94.1% (+18.0%)    16.91 (+6.36) X      1.971 (+1.02)    4.625 (+3.07) X
X = misses the §9.1 pass bar (citation match has no bar of its own); deltas are vs the previous row


## Metrics before and after

Saved results only: `evals/results/*_summary.json` plus the Nebius judge CSVs. Deltas are AFTER minus BEFORE; X marks a missed §9.1 pass bar.

In [2]:
print("faithfulness regressions v3 -> v4:", compare.faithfulness_regressions("v3", "v4"))
for v in ("v3", "v4", "v5"):
    s = compare.load_summary(v)
    print(v, "run cost INR", s["run"]["total_cost_inr"], "| unsupported-claim cases", len(compare.failure_modes(v)["unsupported claim"]))

faithfulness regressions v3 -> v4: [{'case_id': 'F21', 'agent': 'money', 'scenario_type': 'happy', 'unsupported': 'Ask the hospital billing team whether they will send the pre-authorization and for itemised bills.', 'question': 'My baby was admitted to NICU. What should I ask the hospital billing team and insurer?'}]
v3 run cost INR 98.93 | unsupported-claim cases 16
v4 run cost INR 204.97 | unsupported-claim cases 4
v5 run cost INR 179.14 | unsupported-claim cases 11


## Reading the result

- **Faithfulness is the one lever that beat its prediction:** 76.1% to 94.1% (+18 points against +5), clearing the 90% bar. Unsupported-claim cases fell 16 to 4.
- **It was expensive:** p95 latency 10.55 s to 16.91 s (predicted +1 to 2 s), mean cost 0.951 to 1.971 INR and p95 cost 4.625 INR, over the 2 INR bar. Run cost 98.93 to 204.97 INR.
- **Citation match fell 78.1% to 67.1%** (the cause was not isolated here). Only one case, F21, went from faithful in v3 to unfaithful in v4.
- **v5 (round 2) trades some faithfulness back:** v4 to v5 faithfulness 94.1% to 84.1% (nine regressions, listed in notebook 06) in exchange for routing 97.1%, refusal 100%, lower cost (1.722 mean) and slightly lower p95 latency (15.48 s). p95 latency still misses 8 s and p95 cost still misses 2 INR.

## Live run (off)

Re-running costs money (see the cost column) and writes a new LangSmith experiment. Saved results above are what this notebook reports.

In [3]:
RUN_LIVE = False

if RUN_LIVE:
    from dotenv import find_dotenv, load_dotenv

    load_dotenv(find_dotenv(usecwd=True))
    from nestling import graph

    graph.run_experiment("v4")
else:
    print("RUN_LIVE = False: showing saved results only")

RUN_LIVE = False: showing saved results only
